<b> Stream Customers Data From Cloud Files to Delta Lake using Auto Loader
1. Read files from cloud storage using Auto Loader 
2. Transform the dataframe to add the following columns  
   - file path: Cloud file path  
   - ingestion date: Current Timestamp  
3. Write the transformed data stream to Delta Lake Table

<b> 1. Read files using Auto Loader


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customers_schema = StructType(fields=[
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("date_of_birth", DateType()),
    StructField("telephone", StringType()),
    StructField("email", StringType()),
    StructField("member_since", DateType()),
    StructField("created_timestamp", TimestampType())
])

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp

customer_df = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "json")
    .option("cloudfiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
    .option("cloudfiles.inferColumnTypes", "true")
    .option("pathGlobFilter", "customers_2024_*.json")
    .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
    .withColumn("date_of_birth", to_date(col("date_of_birth")))
    .withColumn("member_since", to_date(col("member_since")))
    .withColumn("created_timestamp", to_timestamp(col("created_timestamp")))
)

<b>2. Transform the dataframe to add the following columns  
- file path: Cloud file path  
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_tranformed_df = (
                                customer_df.withColumn("file_path", col("_metadata.file_path"))
                                .withColumn("ingestion_date", current_timestamp())
)

<b> 3. Write the transformed data stream to Delta Table

In [0]:

streaming_query = (
    customers_tranformed_df.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
        .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
%sql
SELECT * FROM gizmobox.bronze.customers_autoloader;